In [0]:
%sql
CREATE OR REPLACE FUNCTION admin.access_control.mask_by_hierarchy(
  val STRING, 
  target_customer_id STRING
)
RETURN IF(
  EXISTS (
    -- Establish User Context
    WITH user_context AS (
        SELECT 
            employee_name,
            LOWER(TRIM(role)) as role_name
        FROM admin.access_control.sales_reporting_hierarchy
        WHERE LOWER(TRIM(email)) = LOWER(TRIM(current_user()))
    ),
    -- Determine if current_user has a path to this specific customer
    access_check AS (
        SELECT 1
        FROM user_context uc
        LEFT JOIN admin.access_control.customer_manager cm
            ON CAST(cm.customer_code AS STRING) = CAST(target_customer_id AS STRING)
        LEFT JOIN admin.access_control.sales_reporting_hierarchy srh_csm 
            ON LOWER(TRIM(cm.customer_success_manager_name)) = LOWER(TRIM(srh_csm.employee_name))
        LEFT JOIN admin.access_control.sales_reporting_hierarchy srh_lead
            ON LOWER(TRIM(srh_csm.manager_name)) = LOWER(TRIM(srh_lead.employee_name))
        WHERE 
            -- Director Access
            uc.role_name IN ('sales director', 'director')
            -- CSM Access
            OR LOWER(TRIM(uc.employee_name)) = LOWER(TRIM(cm.customer_success_manager_name))
            -- Team Lead Access
            OR LOWER(TRIM(uc.employee_name)) = LOWER(TRIM(srh_csm.manager_name))
            -- Assistant Director Access
            OR LOWER(TRIM(uc.employee_name)) = LOWER(TRIM(srh_lead.manager_name))
    )
    SELECT 1 FROM access_check
  ),
  val,
  '***MASKED***'
);

In [0]:
%sql
ALTER TABLE customer.transaction.bronze_transaction_data
SET ROW FILTER admin.access_control.customer_row_filter ON (customer_id);

In [0]:
%sql
